# Imports

In [ ]:
import datetime
import glob
import json
import os
import matplotlib
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import statistics
import scipy.stats as stats
import scipy
from datetime import date
import pylandstats as pls
import cv2 as cv2
from scipy import spatial
import math
from scipy.stats.stats import pearsonr

import plotly.express as px

import statsmodels.formula.api as smf

import matplotlib.ticker as mticker

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)


#!! Set working directory !!#
userpath='C:/Users/peter/Documents/GitHub/HoneybeeNestMorphogenesis'
os.chdir(userpath)

colony_names = ['DD1', 'DD2', 'DD4', 'DD5', 'DD6', 'DD7', 'SH2', 'SH3', 'SH4', 'SH5', 'SH6', 'SH7']

content_names = [
 'background',
 'wood',
 'empty_comb',
 'eggs',
 'larvae',
 'pupae',
 'pollen',
 'nectar',
 'honey',
 'queen_cup',
 'queen_cell',
 'bees',
]
short_names = content_names[:9]

def showmask(mask, ax=None): # show 1 comb mask
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(mask, cmap='gist_stern', vmin=0, vmax=8)
    ax.axis('off')

contenthex = ['#000000', #background
             '#a62040', #wood
             '#404080', #empty_comb
             '#6060bf', #eggs
             '#8080ff', #larvae
             '#9f9f77', #pupae
             '#bfbf0e', #pollen
             '#dfdf87', #nectar
             '#ffffff', #honey
             ]

C:\Users\peter\AppData\Local\Temp\ipykernel_19524\3535885437.py:19: DeprecationWarning: Please import `pearsonr` from the `scipy.stats` namespace; the `scipy.stats.stats` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.stats.stats import pearsonr


# Cell Neighbors

In [ ]:
# Load comb type array
#using aligned
Contents = np.load('CombMasks_Contents_aligned.npy', allow_pickle=True) 

In [ ]:
# make a list of positions around a focal point where we will look to see the neighboring content type
# going clockwise starting at noon
neibpositions = np.array([[1,0],
                   [1,1],
                   [0,1],
                   [-1,1],
                   [-1,0],
                   [-1,-1],
                   [0,-1],
                   [1,-1]])

In [ ]:
class_names = [
 'background',
 'wood',
 'empty_comb',
 'eggs',
 'larvae',
 'pupae',
 'pollen',
 'nectar',
 'honey',
 'queen_cup',
 'queen_cell',
 'bees',
]

In [ ]:
# the interesting content to focus on
contentlist = [3,4,5,6,7,8]

In [ ]:
# using the comb content numpy array

# maxweek=10

side='a'
df_cellneib = pd.DataFrame()
for c in range(len(Contents)):
    print(colony_names[c])
    for w in range(len(Contents[c])):
        for f in range(20):
            mask = Contents[c][w][f]
            ncells = np.count_nonzero(mask > 1)
            
            for focalcontent in contentlist:
                coords = np.argwhere(mask==focalcontent)

                if len(coords)>0:

                    neiblistall = []
                    for focalpos in coords:
                        if (focalpos[0]<mask.shape[0]-1)&(focalpos[1]<mask.shape[1]-1):
                            neiblist = []
                            for neibposition in neibpositions:
                                combpos = focalpos + neibposition
                                neib = mask[combpos[0]][combpos[1]]
                                neiblist.append(neib)
                            neiblistall.append(neiblist)
                    neiblistall = np.array(neiblistall)

                    # make summary dataframe
                    neibsum = len(neiblistall)*len(neiblist)
                    for content in range(len(class_names)):

                        contentcount = np.count_nonzero(neiblistall == content)
                        contentperc = (contentcount/neibsum)*100

                        dftemp = pd.DataFrame()
                        dftemp['colony'] = [colony_names[c]]
                        dftemp['week'] = w+1
                        dftemp['frame'] = [int(f/2)+1]
                        dftemp['side'] = side
                        dftemp['framenum'] = f
                        dftemp['focalcontent'] = class_names[focalcontent]
                        dftemp['nfocalcells'] = len(coords)
                        dftemp['neibcontent'] = class_names[content]
                        dftemp['percniebcontent_onframe'] = (np.count_nonzero(mask==content)/ncells)*100
                        dftemp['neibperc'] = contentperc

                        df_cellneib = pd.concat([df_cellneib,dftemp]).reset_index(drop=True)
                        
            if side=='a':
                side='b'
            else:
                side='a'
df_cellneib

# ~1 hour

DD1
DD2
DD4
DD5
DD6
DD7
SH2
SH3
SH4
SH5
SH6
SH7


,colony,week,frame,side,framenum,focalcontent,nfocalcells,neibcontent,percniebcontent_onframe,neibperc
0,DD1,1,5,a,8,eggs,125,background,537.662338,0.0
1,DD1,1,5,a,8,eggs,125,wood,112.698413,0.0
2,DD1,1,5,a,8,eggs,125,empty_comb,71.284271,43.3
3,DD1,1,5,a,8,eggs,125,eggs,18.037518,55.8
4,DD1,1,5,a,8,eggs,125,larvae,1.010101,0.8
...,...,...,...,...,...,...,...,...,...,...
239551,SH7,18,10,b,19,nectar,2,nectar,0.060588,0.0
239552,SH7,18,10,b,19,nectar,2,honey,0.000000,0.0
239553,SH7,18,10,b,19,nectar,2,queen_cup,0.000000,0.0
239554,SH7,18,10,b,19,nectar,2,queen_cell,0.000000,0.0


In [ ]:
df_cellneib.loc[(df_cellneib.neibcontent == 'background')|(df_cellneib.neibcontent == 'wood'), 'percniebcontent_onframe'] = np.nan
df_cellneib['perc_diff'] = df_cellneib.neibperc - df_cellneib.percniebcontent_onframe
df_cellneib.loc[df_cellneib.percniebcontent_onframe == 0, 'perc_diff'] = np.nan
df_cellneib

,colony,week,frame,side,framenum,focalcontent,nfocalcells,neibcontent,percniebcontent_onframe,neibperc,perc_diff
0,DD1,1,5,a,8,eggs,125,background,NaN,0.0,NaN
1,DD1,1,5,a,8,eggs,125,wood,NaN,0.0,NaN
2,DD1,1,5,a,8,eggs,125,empty_comb,71.284271,43.3,-27.984271
3,DD1,1,5,a,8,eggs,125,eggs,18.037518,55.8,37.762482
4,DD1,1,5,a,8,eggs,125,larvae,1.010101,0.8,-0.210101
...,...,...,...,...,...,...,...,...,...,...,...
239551,SH7,18,10,b,19,nectar,2,nectar,0.060588,0.0,-0.060588
239552,SH7,18,10,b,19,nectar,2,honey,0.000000,0.0,NaN
239553,SH7,18,10,b,19,nectar,2,queen_cup,0.000000,0.0,NaN
239554,SH7,18,10,b,19,nectar,2,queen_cell,0.000000,0.0,NaN


In [ ]:
df_cellneib['frame_reoriented'] = df_cellneib.frame
df_cellneib.loc[(df_cellneib.colony!='DD6')|(df_cellneib.colony!='DD7')|(df_cellneib.colony!='SH4'), 'frame_reoriented'] = 11 - df_cellneib.frame

df_cellneib['seasonal_phase'] = 'spring_growth'
df_cellneib.loc[(df_cellneib.week>11)&(df_cellneib.week<17), 'seasonal_phase'] = 'summer_dearth'
df_cellneib.loc[df_cellneib.week>16, 'seasonal_phase'] = 'fall_flow'

df_cellneib['treatment'] = df_cellneib['colony'].str[:2]

df_cellneib

,colony,week,frame,side,framenum,focalcontent,nfocalcells,neibcontent,percniebcontent_onframe,neibperc,perc_diff,frame_reoriented,seasonal_phase,treatment
0,DD1,1,5,a,8,eggs,125,background,NaN,0.0,NaN,6,spring_growth,DD
1,DD1,1,5,a,8,eggs,125,wood,NaN,0.0,NaN,6,spring_growth,DD
2,DD1,1,5,a,8,eggs,125,empty_comb,71.284271,43.3,-27.984271,6,spring_growth,DD
3,DD1,1,5,a,8,eggs,125,eggs,18.037518,55.8,37.762482,6,spring_growth,DD
4,DD1,1,5,a,8,eggs,125,larvae,1.010101,0.8,-0.210101,6,spring_growth,DD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239551,SH7,18,10,b,19,nectar,2,nectar,0.060588,0.0,-0.060588,1,fall_flow,SH
239552,SH7,18,10,b,19,nectar,2,honey,0.000000,0.0,NaN,1,fall_flow,SH
239553,SH7,18,10,b,19,nectar,2,queen_cup,0.000000,0.0,NaN,1,fall_flow,SH
239554,SH7,18,10,b,19,nectar,2,queen_cell,0.000000,0.0,NaN,1,fall_flow,SH


In [ ]:
# Add date and true week

os.chdir(userpath + r'Box\3D_nest_2021\CombContents_PRM')
df_trueweek = pd.read_csv('df_trueweek.csv')
df_cellneib = pd.read_csv('df_cellneib.csv')

df_cellneib = pd.merge(df_cellneib, df_trueweek, how="left", on=["colony", "week"]).reset_index(drop=True)
df_cellneib = df_cellneib[df_cellneib.columns.tolist()[:1] + ['date', 'true_week'] + df_cellneib.columns.tolist()[1:-2]] # reorder columns
df_cellneib

,colony,date,true_week,week,frame,side,framenum,focalcontent,nfocalcells,neibcontent,percniebcontent_onframe,neibperc,perc_diff,frame_reoriented,seasonal_phase,treatment
0,DD1,20210412,1,1,5,a,8,eggs,125,background,NaN,0.0,NaN,6,spring_growth,DD
1,DD1,20210412,1,1,5,a,8,eggs,125,wood,NaN,0.0,NaN,6,spring_growth,DD
2,DD1,20210412,1,1,5,a,8,eggs,125,empty_comb,71.284271,43.3,-27.984271,6,spring_growth,DD
3,DD1,20210412,1,1,5,a,8,eggs,125,eggs,18.037518,55.8,37.762482,6,spring_growth,DD
4,DD1,20210412,1,1,5,a,8,eggs,125,larvae,1.010101,0.8,-0.210101,6,spring_growth,DD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239551,SH7,20210930,25,18,10,b,19,nectar,2,nectar,0.060588,0.0,-0.060588,1,fall_flow,SH
239552,SH7,20210930,25,18,10,b,19,nectar,2,honey,0.000000,0.0,NaN,1,fall_flow,SH
239553,SH7,20210930,25,18,10,b,19,nectar,2,queen_cup,0.000000,0.0,NaN,1,fall_flow,SH
239554,SH7,20210930,25,18,10,b,19,nectar,2,queen_cell,0.000000,0.0,NaN,1,fall_flow,SH


In [ ]:
# Save CSV

df_cellneib.to_csv('df_cellneib.csv', index=False)  

# Cell Future Self 

In [ ]:
# Load comb type array
# USING ALIGNED
Contents = np.load('CombMasks_Contents_aligned.npy', allow_pickle=True) 

class_names = [
 'background',
 'wood',
 'empty_comb',
 'eggs',
 'larvae',
 'pupae',
 'pollen',
 'nectar',
 'honey',
 'queen_cup',
 'queen_cell',
 'bees',
]

In [ ]:
# the interesting content to focus on
contentlist = [2,3,4,5,6,7,8]

In [ ]:

side='a'
df_cellfuture = pd.DataFrame()
for c in range(len(Contents)):
    print(colony_names[c])
    for w in range(len(Contents[c])-1):
        for f in range(20):

            mask0 = Contents[c][w][f]
            mask1 = Contents[c][w+1][f]
            ncells = np.count_nonzero(mask0 > 1)

            for focalcontent in contentlist:

                coords = np.argwhere(mask0==focalcontent)

                if len(coords)>0:

                    selflist = mask1[coords[:,0],coords[:,1]]

                    for content in contentlist:
                        contentcount = np.count_nonzero(selflist == content)
                        contentperc = (contentcount/len(selflist))*100

                        dftemp = pd.DataFrame()
                        dftemp['colony'] = [colony_names[c]]
                        dftemp['week'] = w+1
                        dftemp['frame'] = [int(f/2)+1]
                        dftemp['side'] = side
                        dftemp['framenum'] = f
                        dftemp['focalcontent'] = class_names[focalcontent]
                        dftemp['nfocalcells'] = len(coords)
                        dftemp['nextself'] = class_names[content]
                        dftemp['nextselfcount'] = contentcount
                        # dftemp['percselfcontent_onframe'] = (np.count_nonzero(mask0==content)/ncells)*100
                        dftemp['nextperc'] = contentperc

                        df_cellfuture = pd.concat([df_cellfuture,dftemp]).reset_index(drop=True)

            if side=='a':
                side='b'
            else:
                side='a'

df_cellfuture

# ~24m

DD1
DD2
DD4
DD5
DD6
DD7
SH2
SH3
SH4
SH5
SH6
SH7


,colony,week,frame,side,framenum,focalcontent,nfocalcells,nextself,nextselfcount,nextperc
0,DD1,1,5,a,8,empty_comb,494,empty_comb,149,30.161943
1,DD1,1,5,a,8,empty_comb,494,eggs,9,1.821862
2,DD1,1,5,a,8,empty_comb,494,larvae,245,49.595142
3,DD1,1,5,a,8,empty_comb,494,pupae,0,0.000000
4,DD1,1,5,a,8,empty_comb,494,pollen,3,0.607287
...,...,...,...,...,...,...,...,...,...,...
163690,SH7,17,10,b,19,nectar,1,larvae,0,0.000000
163691,SH7,17,10,b,19,nectar,1,pupae,0,0.000000
163692,SH7,17,10,b,19,nectar,1,pollen,1,100.000000
163693,SH7,17,10,b,19,nectar,1,nectar,0,0.000000


In [ ]:
df_cellfuture['frame_reoriented'] = df_cellfuture.frame
df_cellfuture.loc[(df_cellfuture.colony!='DD6')|(df_cellfuture.colony!='DD7')|(df_cellfuture.colony!='SH4'), 'frame_reoriented'] = 11 - df_cellfuture.frame

df_cellfuture['seasonal_phase'] = 'spring_growth'
df_cellfuture.loc[(df_cellfuture.week>11)&(df_cellfuture.week<17), 'seasonal_phase'] = 'summer_dearth'
df_cellfuture.loc[df_cellfuture.week>16, 'seasonal_phase'] = 'fall_flow'

df_cellfuture['treatment'] = df_cellfuture['colony'].str[:2]

In [ ]:
# Add date and true week

os.chdir(userpath + r'Box\3D_nest_2021\CombContents_PRM')
df_trueweek = pd.read_csv('df_trueweek.csv')
df_cellfuture = pd.read_csv('df_cellfuture.csv')

df_cellfuture = pd.merge(df_cellfuture, df_trueweek, how="left", on=["colony", "week"]).reset_index(drop=True)
df_cellfuture = df_cellfuture[df_cellfuture.columns.tolist()[:1] + ['date', 'true_week'] + df_cellfuture.columns.tolist()[1:-2]] # reorder columns
df_cellfuture

,colony,date,true_week,week,frame,side,framenum,focalcontent,nfocalcells,nextself,nextselfcount,nextperc,frame_reoriented,seasonal_phase,treatment
0,DD1,20210412,1,1,5,a,8,empty_comb,494,empty_comb,149,30.161943,6,spring_growth,DD
1,DD1,20210412,1,1,5,a,8,empty_comb,494,eggs,9,1.821862,6,spring_growth,DD
2,DD1,20210412,1,1,5,a,8,empty_comb,494,larvae,245,49.595142,6,spring_growth,DD
3,DD1,20210412,1,1,5,a,8,empty_comb,494,pupae,0,0.000000,6,spring_growth,DD
4,DD1,20210412,1,1,5,a,8,empty_comb,494,pollen,3,0.607287,6,spring_growth,DD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163690,SH7,20210923,24,17,10,b,19,nectar,1,larvae,0,0.000000,1,fall_flow,SH
163691,SH7,20210923,24,17,10,b,19,nectar,1,pupae,0,0.000000,1,fall_flow,SH
163692,SH7,20210923,24,17,10,b,19,nectar,1,pollen,1,100.000000,1,fall_flow,SH
163693,SH7,20210923,24,17,10,b,19,nectar,1,nectar,0,0.000000,1,fall_flow,SH


In [ ]:
# Save CSV

df_cellfuture.to_csv('df_cellfuture.csv', index=False)  

In [ ]:
df_cellfuture

,colony,date,true_week,week,frame,side,framenum,focalcontent,nfocalcells,nextself,nextselfcount,nextperc,frame_reoriented,seasonal_phase,treatment
0,DD1,20210412,1,1,5,a,8,empty_comb,494,empty_comb,149,30.161943,6,spring_growth,DD
1,DD1,20210412,1,1,5,a,8,empty_comb,494,eggs,9,1.821862,6,spring_growth,DD
2,DD1,20210412,1,1,5,a,8,empty_comb,494,larvae,245,49.595142,6,spring_growth,DD
3,DD1,20210412,1,1,5,a,8,empty_comb,494,pupae,0,0.000000,6,spring_growth,DD
4,DD1,20210412,1,1,5,a,8,empty_comb,494,pollen,3,0.607287,6,spring_growth,DD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163690,SH7,20210923,24,17,10,b,19,nectar,1,larvae,0,0.000000,1,fall_flow,SH
163691,SH7,20210923,24,17,10,b,19,nectar,1,pupae,0,0.000000,1,fall_flow,SH
163692,SH7,20210923,24,17,10,b,19,nectar,1,pollen,1,100.000000,1,fall_flow,SH
163693,SH7,20210923,24,17,10,b,19,nectar,1,nectar,0,0.000000,1,fall_flow,SH


# Cluster metric (nearest neighb)

In [ ]:
# Load arrays and functions

# aligned and CROPPED images

ContentsCrop = np.load('CombMasks_Contents_crop.npy', allow_pickle=True) 


In [ ]:
# Create dataset compared to random distribution
# AVG OF SEVERAL RANDOM SAMPLES

from scipy import spatial

nsamples = 100 # number of random samples to take
thres = 10 # threshold for number of cells of content type on a given frame to incluse in anaysis

appended=[] # list to append dataframes
side='a' # start with side a, will alternate to side b
for c in range(12): # for each colony
    for w in range(len(ContentsCrop[c])): # for each week
        for f in range(20): # for each frame
            mask = ContentsCrop[c][w][f] # load mask

            combcoords = np.argwhere(mask>1) # coordinates of all comb

            for content in range(3,9): # for each content type of interest
                contentcoords = np.argwhere(mask==content)  #  coordinates of content type

                if len(contentcoords)>thres:

                    # make random distribution of content
                    randlist = [] # list to store random nearest neighbor distances
                    for n in range(nsamples): # for each random sample
                        randindex = np.random.choice(len(combcoords), len(contentcoords), replace=False) # get random indices from comb the size of contents on the comb
                        randcoords = combcoords[randindex] # get random coordinates
                        randtree = spatial.cKDTree(randcoords) # create spatial tree for random coordinates
                        rd, ri = randtree.query(randcoords, k=2) # find nearest neighbor for each random coordinate
                        randnearestmean = np.mean(rd[:,1]) # average distance to nearest neighbor for random sample
                        randlist.append(randnearestmean)
                    randnearestmean = np.mean(randlist) # average distance to nearest neighbor from all random samples

                    # actual nearest mean
                    tree = spatial.cKDTree(contentcoords) # create spatial tree for content coordinates
                    d, i = tree.query(contentcoords, k=2) # find nearest neighbor for each content coordinate
                    nearestmean = np.mean(d[:,1]) # average distance to nearest neighbor for actual content

                    # Monte Carlo p-value for distinction from random distribution
                    p_value = (np.sum(np.array(randlist) <= nearestmean) + 1) / (nsamples + 1)

                    # create dataframe to store results
                    temp_counts = pd.DataFrame()
                    temp_counts['colony'] = [colony_names[c]]
                    temp_counts['week'] = [w+1]
                    temp_counts['frame'] = [int(f/2)+1]
                    temp_counts['side'] = [side]
                    temp_counts['treatment'] = [colony_names[c][:2]]
                    temp_counts['contentid'] = [content]
                    temp_counts['content'] = [content_names[content]]
                    temp_counts['count'] = [len(contentcoords)]
                    temp_counts['rand_nearestneibavg'] = [randnearestmean]
                    temp_counts['nearestneibavg'] = [nearestmean]
                    temp_counts['combcount'] = [len(combcoords)]
                    temp_counts['p_value'] = [p_value]

                    appended.append(temp_counts)

            if side=='a': # alternate sides
                side='b'
            else:
                side='a'

        print('colony: ' + colony_names[c] + ', week: ' + str(w+1).zfill(2), end='\r') # progress indicator

df_nearestdist = pd.concat(appended).reset_index(drop=True) # combine all dataframes
df_nearestdist
# ~10min

,colony,week,frame,side,treatment,contentid,content,count,rand_nearestneibavg,nearestneibavg,combcount,p_value
0,DD1,1,5,a,DD,3,eggs,125,1.383787,1.042405,693,0.009901
1,DD1,1,5,a,DD,7,nectar,59,1.922707,1.033898,693,0.009901
2,DD1,1,5,b,DD,3,eggs,128,1.367748,1.027229,687,0.009901
3,DD1,1,5,b,DD,6,pollen,14,3.934240,1.336300,687,0.009901
4,DD1,1,5,b,DD,7,nectar,54,2.000753,1.007671,687,0.009901
...,...,...,...,...,...,...,...,...,...,...,...,...
16187,SH7,18,8,b,SH,6,pollen,52,4.303284,2.318096,3292,0.009901
16188,SH7,18,9,a,SH,6,pollen,67,3.770076,1.796521,3202,0.009901
16189,SH7,18,9,b,SH,6,pollen,59,3.914196,1.869410,3193,0.009901
16190,SH7,18,10,a,SH,6,pollen,83,3.417093,1.644732,3309,0.009901


In [ ]:
df_nearestdist['content_density'] = df_nearestdist['count'] / df_nearestdist['combcount']
df_nearestdist['diff'] = df_nearestdist['rand_nearestneibavg'] - df_nearestdist['nearestneibavg']
df_nearestdist['perc_diff'] = round((df_nearestdist['diff'] / df_nearestdist['rand_nearestneibavg'])*100,2)

# nearest neighbor metric goes from 0 (randomly distributed) to 1 (tightly clustered)
df_nearestdist['dispersion_metric'] = 1 - ((df_nearestdist['nearestneibavg'] - 1)/(df_nearestdist['rand_nearestneibavg'] - 1)) # value = 1: all cells have at least one immidiate neighbor of the same content; value = 0 is no different than randomly distributed

df_nearestdist.head(50)


,colony,week,frame,side,treatment,contentid,content,count,rand_nearestneibavg,nearestneibavg,combcount,p_value,content_density,diff,perc_diff,dispersion_metric
0,DD1,1,5,a,DD,3,eggs,125,1.383787,1.042405,693,0.009901,0.180375,0.341382,24.67,0.889510
1,DD1,1,5,a,DD,7,nectar,59,1.922707,1.033898,693,0.009901,0.085137,0.888809,46.23,0.963262
2,DD1,1,5,b,DD,3,eggs,128,1.367748,1.027229,687,0.009901,0.186317,0.340519,24.90,0.925958
3,DD1,1,5,b,DD,6,pollen,14,3.934240,1.336300,687,0.009901,0.020378,2.597940,66.03,0.885388
4,DD1,1,5,b,DD,7,nectar,54,2.000753,1.007671,687,0.009901,0.078603,0.993083,49.64,0.992335
5,DD1,1,6,a,DD,3,eggs,253,1.282465,1.021681,1165,0.009901,0.217167,0.260785,20.33,0.923244
6,DD1,1,6,a,DD,4,larvae,168,1.492873,1.035565,1165,0.009901,0.144206,0.457308,30.63,0.927842
7,DD1,1,6,a,DD,6,pollen,54,2.530148,1.121170,1165,0.009901,0.046352,1.408978,55.69,0.920811
8,DD1,1,6,a,DD,7,nectar,238,1.311790,1.009423,1165,0.009901,0.204292,0.302367,23.05,0.969778
9,DD1,1,6,b,DD,3,eggs,262,1.264583,1.042119,1174,0.009901,0.223169,0.222464,17.59,0.840811


In [ ]:
# how many instances of random distribution (p>0.05) in DDs?

df = df_nearestdist[(df_nearestdist['p_value']>0.05) & (df_nearestdist['treatment']=='DD')]

df

,colony,date,true_week,week,frame,side,treatment,contentid,content,count,rand_nearestneibavg,nearestneibavg,combcount,p_value,content_density,diff,perc_diff,dispersion_metric
395,DD1,20210508,5,5,9,b,DD,7,nectar,15,7.962045,6.062555,3107,0.059406,0.004828,1.899490,23.86,0.272835
976,DD1,20210728,16,13,9,b,DD,6,pollen,11,10.095792,12.814710,3373,0.910891,0.003261,-2.718917,-26.93,-0.298920
2420,DD2,20210826,20,15,6,a,DD,6,pollen,16,8.112163,6.030319,3425,0.059406,0.004672,2.081844,25.66,0.292716
2421,DD2,20210826,20,15,6,a,DD,7,nectar,12,9.567639,7.321664,3425,0.099010,0.003504,2.245974,23.47,0.262146
2533,DD2,20210923,24,17,7,b,DD,6,pollen,13,9.215261,7.999862,3430,0.198020,0.003790,1.215399,13.19,0.147944
3562,DD4,20210826,20,15,10,a,DD,7,nectar,20,7.361374,5.806768,3383,0.079208,0.005912,1.554606,21.12,0.244382
3821,DD4,20220113,40,21,4,a,DD,6,pollen,12,9.488738,6.100421,3280,0.059406,0.003659,3.388317,35.71,0.399154
5010,DD5,20210826,20,15,9,a,DD,6,pollen,21,7.216100,5.774363,3404,0.099010,0.006169,1.441737,19.98,0.231936
5060,DD5,20210910,22,16,7,a,DD,6,pollen,20,7.295224,6.395501,3401,0.158416,0.005881,0.899723,12.33,0.142922
5118,DD5,20210923,24,17,6,a,DD,4,larvae,18,7.822074,7.368165,3400,0.306931,0.005294,0.453909,5.80,0.066535


In [ ]:
# Comb face weighted avg
# take weighted average for the comb face
def weighted_average(dataframe, value, weight):
    val = dataframe[value]
    wt = dataframe[weight]
    return (val * wt).sum() / wt.sum()
df_nearestdist_sum = pd.DataFrame(df_nearestdist.groupby(['colony', 'treatment', 'week', 'frame', 'side']).apply(weighted_average, 
                                     'dispersion_metric', 'count')).reset_index()
df_nearestdist_sum.rename(columns={0:'dispersion_metric'}, inplace=True)
df_nearestdist_sum

,colony,treatment,week,frame,side,dispersion_metric
0,DD1,DD,1,5,a,0.913159
1,DD1,DD,1,5,b,0.941348
2,DD1,DD,1,6,a,0.939676
3,DD1,DD,1,6,b,0.893401
4,DD1,DD,1,7,a,0.940265
...,...,...,...,...,...,...
4138,SH7,SH,18,8,b,0.600974
4139,SH7,SH,18,9,a,0.712455
4140,SH7,SH,18,9,b,0.701664
4141,SH7,SH,18,10,a,0.733261


In [ ]:
# Add date and true week

df_trueweek = pd.read_csv('df_trueweek.csv')
df_nearestdist = pd.read_csv('df_nearestdist.csv')
df_nearestdist_sum = pd.read_csv('df_nearestdist_sum.csv')  

df_nearestdist = pd.merge(df_nearestdist, df_trueweek, how="left", on=["colony", "week"]).reset_index(drop=True)
df_nearestdist = df_nearestdist[df_nearestdist.columns.tolist()[:1] + ['date', 'true_week'] + df_nearestdist.columns.tolist()[1:-2]] # reorder columns
df_nearestdist_sum = pd.merge(df_nearestdist_sum, df_trueweek, how="left", on=["colony", "week"]).reset_index(drop=True)
df_nearestdist_sum = df_nearestdist_sum[df_nearestdist_sum.columns.tolist()[:1] + ['date', 'true_week'] + df_nearestdist_sum.columns.tolist()[1:-2]] # reorder columns

df_nearestdist

,colony,date,true_week,week,frame,side,treatment,contentid,content,count,rand_nearestneibavg,nearestneibavg,combcount,p_value,content_density,diff,perc_diff,dispersion_metric
0,DD1,20210412,1,1,5,a,DD,3,eggs,125,1.383787,1.042405,693,0.009901,0.180375,0.341382,24.67,0.889510
1,DD1,20210412,1,1,5,a,DD,7,nectar,59,1.922707,1.033898,693,0.009901,0.085137,0.888809,46.23,0.963262
2,DD1,20210412,1,1,5,b,DD,3,eggs,128,1.367748,1.027229,687,0.009901,0.186317,0.340519,24.90,0.925958
3,DD1,20210412,1,1,5,b,DD,6,pollen,14,3.934240,1.336300,687,0.009901,0.020378,2.597940,66.03,0.885388
4,DD1,20210412,1,1,5,b,DD,7,nectar,54,2.000753,1.007671,687,0.009901,0.078603,0.993083,49.64,0.992335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16187,SH7,20210930,25,18,8,b,SH,6,pollen,52,4.303284,2.318096,3292,0.009901,0.015796,1.985188,46.13,0.600974
16188,SH7,20210930,25,18,9,a,SH,6,pollen,67,3.770076,1.796521,3202,0.009901,0.020924,1.973555,52.35,0.712455
16189,SH7,20210930,25,18,9,b,SH,6,pollen,59,3.914196,1.869410,3193,0.009901,0.018478,2.044786,52.24,0.701664
16190,SH7,20210930,25,18,10,a,SH,6,pollen,83,3.417093,1.644732,3309,0.009901,0.025083,1.772361,51.87,0.733261


In [ ]:
# Save CSV

df_nearestdist.to_csv('df_nearestdist.csv', index=False)  
df_nearestdist_sum.to_csv('df_nearestdist_sum.csv', index=False)  

## Nest level (3D)

In [ ]:
# Load and import

# aligned and CROPPED images

ContentsCrop = np.load('CombMasks_Contents_crop.npy', allow_pickle=True) 
df_mergedinsp = pd.read_csv('df_mergedlandscape.csv')



In [ ]:
# functions

def flipmasks(masks):
    # flip every other frame in a whole nest masks (takes masks for whole nest at one timepoint)
    flippedmasks = np.copy(masks)
    flippedmasks[1::2, :, :] = flippedmasks[1::2, :, ::-1] #flip every other frame
    return flippedmasks

def arrange_nest(masks, colonyname, w, data, flip=True, shift=False):
    # Arange 3d nest "masks" into the position and orientation "w" week

    if flip:
        masks = flipmasks(masks) # flip every other mask in nest 
    if shift:
        masks = shiftmasks(masks) # shift every other mask to align 

    data = data[(data['colony']==colonyname)&
            (data['week']==w+1)&
            (data['side']=='a')
            ] 
    data = data.sort_values('position')
    frameorder = data.frame.tolist()
    inverted = data.invert_frame.tolist()
    masksoriented=[]
    for f in range(len(data)):
        if inverted[f]=='y':
            b = int((frameorder[f]-1)*2) + 1
            maskb = masks[b]
            maskb = np.flip(maskb, 1)
            masksoriented.append(maskb)

            a = b-1
            maska = masks[a]
            maska = np.flip(maska, 1)
            masksoriented.append(maska)
        else:
            a = int((frameorder[f]-1)*2)
            maska = masks[a]
            masksoriented.append(maska)

            b = a+1
            maskb = masks[b]
            masksoriented.append(maskb)

    masksoriented = np.array(masksoriented)
    
    return masksoriented

In [ ]:
# 3D - Whole nest at a time
# AVG OF SEVERAL RANDOM SAMPLES

from scipy import spatial

data = df_mergedinsp
thres = 50 # threshold for number of cells of content type on a given nest to include in analysis
nsamples = 100 # number of random samples to take   

appended=[] # list to append dataframes
for c in range(12): # for each colony
    for w in range(len(ContentsCrop[c])): # for each week
        masks = ContentsCrop[c][w] # load masks for whole nest
        masks = arrange_nest(masks, colony_names[c], w, data) # arrange masks for the week based on their postion and orientation (especially important for shuffled colonies)
        combcoords = np.argwhere(masks>1) # coordinates of all comb in nest

        for content in range(3,9): # for each content type of interest
            contentcoords = np.argwhere(masks==content) # coordinates of content type 

            if len(contentcoords)>thres: # threshold for number of cells of content type on a given nest to include in analysis

                # make random distribution of content
                randlist = [] # list to store random nearest neighbor distances
                for n in range(nsamples): # for each random sample
                    randindex = np.random.choice(len(combcoords), len(contentcoords), replace=False) # get random indices from nest the size of contents in the nest
                    randcoords = combcoords[randindex] # get random coordinates
                    randtree = spatial.cKDTree(randcoords) # create spatial tree for random coordinates
                    rd, ri = randtree.query(randcoords, k=2) # find nearest neighbor for each random coordinate
                    randnearestmean = np.mean(rd[:,1]) # average distance to nearest neighbor for random sample
                    randlist.append(randnearestmean)
                randnearestmean = np.mean(randlist) # average distance to nearest neighbor from all random samples

                # actual nearest mean
                tree = spatial.cKDTree(contentcoords) # create spatial tree for content coordinates
                d, i = tree.query(contentcoords, k=2) # find nearest neighbor for each content coordinate
                nearestmean = np.mean(d[:,1]) # average distance to nearest neighbor for actual content

                # Monte Carlo p-value for distinction from random distribution
                p_value = (np.sum(np.array(randlist) <= nearestmean) + 1) / (nsamples + 1)

                # create dataframe to store results
                temp_counts = pd.DataFrame()
                temp_counts['colony'] = [colony_names[c]]
                temp_counts['week'] = [w+1]
                temp_counts['treatment'] = [colony_names[c][:2]]
                temp_counts['contentid'] = [content]
                temp_counts['content'] = [content_names[content]]
                temp_counts['count'] = [len(contentcoords)]
                temp_counts['rand_nearestneibavg'] = [randnearestmean]
                temp_counts['nearestneibavg'] = [nearestmean]
                temp_counts['combcount'] = [len(combcoords)]
                temp_counts['p_value'] = [p_value]

                appended.append(temp_counts)

        print('colony: ' + colony_names[c] + ', week: ' + str(w+1).zfill(2), end='\r') # progress indicator

df_nearestdist_nest = pd.concat(appended).reset_index(drop=True)
df_nearestdist_nest

# ~12mins

,colony,week,treatment,contentid,content,count,rand_nearestneibavg,nearestneibavg,combcount,p_value
0,DD1,1,DD,3,eggs,2167,1.208392,1.018081,14690,0.009901
1,DD1,1,DD,4,larvae,3406,1.102417,1.005801,14690,0.009901
2,DD1,1,DD,6,pollen,1365,1.360028,1.022677,14690,0.009901
3,DD1,1,DD,7,nectar,1720,1.278545,1.019601,14690,0.009901
4,DD1,1,DD,8,honey,182,2.604333,1.043065,14690,0.009901
...,...,...,...,...,...,...,...,...,...,...
1230,SH7,17,SH,5,pupae,278,3.672268,2.095061,65720,0.009901
1231,SH7,17,SH,6,pollen,3035,1.647731,1.188346,65720,0.009901
1232,SH7,17,SH,7,nectar,1648,2.000101,1.181184,65720,0.009901
1233,SH7,18,SH,5,pupae,93,5.422978,3.103698,65632,0.009901


In [ ]:
df_nearestdist_nest['content_density'] = df_nearestdist_nest['count'] / df_nearestdist_nest['combcount']
df_nearestdist_nest['diff'] = df_nearestdist_nest['rand_nearestneibavg'] - df_nearestdist_nest['nearestneibavg']
df_nearestdist_nest['perc_diff'] = round((df_nearestdist_nest['diff'] / df_nearestdist_nest['rand_nearestneibavg'])*100,2)

# nearest neighbor metric goes from 0 (randomly distributed) to 1 (tightly clustered)
df_nearestdist_nest['dispersion_metric'] = 1 - ((df_nearestdist_nest['nearestneibavg'] - 1)/(df_nearestdist_nest['rand_nearestneibavg'] - 1)) # value = 1: all cells have at least one neighbor of the same content; value = 0 is no different than randomly distributed

df_nearestdist_nest.head(20)

,colony,week,treatment,contentid,content,count,rand_nearestneibavg,nearestneibavg,combcount,p_value,content_density,diff,perc_diff,dispersion_metric
0,DD1,1,DD,3,eggs,2167,1.208392,1.018081,14690,0.009901,0.147515,0.190311,15.75,0.913234
1,DD1,1,DD,4,larvae,3406,1.102417,1.005801,14690,0.009901,0.231858,0.096616,8.76,0.943364
2,DD1,1,DD,6,pollen,1365,1.360028,1.022677,14690,0.009901,0.092920,0.337351,24.80,0.937013
3,DD1,1,DD,7,nectar,1720,1.278545,1.019601,14690,0.009901,0.117086,0.258945,20.25,0.929631
4,DD1,1,DD,8,honey,182,2.604333,1.043065,14690,0.009901,0.012389,1.561268,59.95,0.973157
5,DD1,2,DD,3,eggs,2377,1.331564,1.025930,24218,0.009901,0.098150,0.305635,22.95,0.921796
6,DD1,2,DD,4,larvae,4085,1.169037,1.011565,24218,0.009901,0.168676,0.157472,13.47,0.931584
7,DD1,2,DD,5,pupae,5181,1.115629,1.002373,24218,0.009901,0.213932,0.113256,10.15,0.979475
8,DD1,2,DD,6,pollen,1517,1.513452,1.045507,24218,0.009901,0.062639,0.467945,30.92,0.911371
9,DD1,2,DD,7,nectar,1715,1.459922,1.083278,24218,0.009901,0.070815,0.376644,25.80,0.818929


In [ ]:
# Comb face weighted avg
# take weighted average for the comb face
def weighted_average(dataframe, value, weight):
    val = dataframe[value]
    wt = dataframe[weight]
    return (val * wt).sum() / wt.sum()
df_nearestdist_nest_sum = pd.DataFrame(df_nearestdist_nest.groupby(['colony', 'treatment', 'week']).apply(weighted_average, 
                                     'dispersion_metric', 'count')).reset_index()
df_nearestdist_nest_sum.rename(columns={0:'dispersion_metric'}, inplace=True)
df_nearestdist_nest_sum

,colony,treatment,week,dispersion_metric
0,DD1,DD,1,0.932939
1,DD1,DD,2,0.935206
2,DD1,DD,3,0.875518
3,DD1,DD,4,0.818681
4,DD1,DD,5,0.848843
...,...,...,...,...
231,SH7,SH,14,0.825128
232,SH7,SH,15,0.866398
233,SH7,SH,16,0.812793
234,SH7,SH,17,0.771151


In [ ]:
# Add date and true week

os.chdir(userpath + r'Box\3D_nest_2021\CombContents_PRM')
df_trueweek = pd.read_csv('df_trueweek.csv')
df_nearestdist_nest = pd.read_csv('df_nearestdist_nest.csv')
df_nearestdist_nest_sum = pd.read_csv('df_nearestdist_nest_sum.csv')  

df_nearestdist_nest = pd.merge(df_nearestdist_nest, df_trueweek, how="left", on=["colony", "week"]).reset_index(drop=True)
df_nearestdist_nest = df_nearestdist_nest[df_nearestdist_nest.columns.tolist()[:1] + ['date', 'true_week'] + df_nearestdist_nest.columns.tolist()[1:-2]] # reorder columns
df_nearestdist_nest_sum = pd.merge(df_nearestdist_nest_sum, df_trueweek, how="left", on=["colony", "week"]).reset_index(drop=True)
df_nearestdist_nest_sum = df_nearestdist_nest_sum[df_nearestdist_nest_sum.columns.tolist()[:1] + ['date', 'true_week'] + df_nearestdist_nest_sum.columns.tolist()[1:-2]] # reorder columns

df_nearestdist

,colony,date,true_week,week,frame,side,treatment,contentid,content,count,rand_nearestneibavg,nearestneibavg,combcount,p_value,content_density,diff,perc_diff,dispersion_metric
0,DD1,20210412,1,1,5,a,DD,3,eggs,125,1.383787,1.042405,693,0.009901,0.180375,0.341382,24.67,0.889510
1,DD1,20210412,1,1,5,a,DD,7,nectar,59,1.922707,1.033898,693,0.009901,0.085137,0.888809,46.23,0.963262
2,DD1,20210412,1,1,5,b,DD,3,eggs,128,1.367748,1.027229,687,0.009901,0.186317,0.340519,24.90,0.925958
3,DD1,20210412,1,1,5,b,DD,6,pollen,14,3.934240,1.336300,687,0.009901,0.020378,2.597940,66.03,0.885388
4,DD1,20210412,1,1,5,b,DD,7,nectar,54,2.000753,1.007671,687,0.009901,0.078603,0.993083,49.64,0.992335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16187,SH7,20210930,25,18,8,b,SH,6,pollen,52,4.303284,2.318096,3292,0.009901,0.015796,1.985188,46.13,0.600974
16188,SH7,20210930,25,18,9,a,SH,6,pollen,67,3.770076,1.796521,3202,0.009901,0.020924,1.973555,52.35,0.712455
16189,SH7,20210930,25,18,9,b,SH,6,pollen,59,3.914196,1.869410,3193,0.009901,0.018478,2.044786,52.24,0.701664
16190,SH7,20210930,25,18,10,a,SH,6,pollen,83,3.417093,1.644732,3309,0.009901,0.025083,1.772361,51.87,0.733261


In [ ]:
# Save CSV

df_nearestdist_nest.to_csv('df_nearestdist_nest.csv', index=False)  
df_nearestdist_nest_sum.to_csv('df_nearestdist_nest_sum.csv', index=False)  